<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/PreferredAI/tutorials/blob/master/recommender-systems/11_next_item_recommendation.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/PreferredAI/tutorials/blob/master/recommender-systems/11_next_item_recommendation.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

# Session-based Next-Item Recommendation

Many real-world recommendation settings have **no long-term user profile**: an
anonymous visitor lands on a site, clicks a few items, and we must predict what
they will interact with *next* — all from the current session alone. This is the
**session-based next-item** task.

Cornac ships a family of sequential models for this task, all sharing the same
interface (`NextItemRecommender`) and the same training substrate (a session
iterator, a shared set of ranking losses, and best-on-validation selection):

| Model | Encoder | Year |
|---|---|---|
| **GRU4Rec** | GRU (recurrent) | 2015 |
| **FPMC** | factorized Markov chain | 2010 |
| **SASRec** | causal self-attention | 2018 |
| **BERT4Rec** | bidirectional Transformer (BERT) | 2019 |
| **GPT2Rec** | causal Transformer (GPT-2) | 2021 |

In this tutorial we (1) train all five on the **Diginetica** dataset, and (2) show
how the **choice of loss function** interacts with each model, using a pre-computed
hyperparameter sweep.

## 1. Setup

In [1]:
!pip install --quiet cornac==2.5.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.9/29.9 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 8.3 MB/s eta 0:00:00


In [2]:
import sys

import numpy as np
import pandas as pd
import torch

import cornac
from cornac.datasets import diginetica
from cornac.eval_methods import NextItemEvaluation
from cornac.metrics import MRR, NDCG, Recall
from cornac.models import BERT4Rec, FPMC, GPT2Rec, GRU4Rec, SASRec, SPop
from cornac.utils import cache

SEED = 123
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"System version : {sys.version.split()[0]}")
print(f"Cornac version : {cornac.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device         : {DEVICE}")

System version : 3.12.13
Cornac version : 2.5.0
PyTorch version: 2.11.0+cu128
Device         : cuda:0


## 2. The Diginetica dataset

`cornac.datasets.diginetica` exposes a curated session-based split of the
[CIKM Cup 2016 Diginetica](https://competitions.codalab.org/forums/7901/1941/)
e-commerce click logs. Each user contributes one held-out session to validation
and one to test (the rest is training). We load it in the **session-based** mode
(the default), where a model only sees the *current* session's history.

`NextItemEvaluation.from_splits` wraps the three splits and an item-ranking
protocol: for every test session it scores all known items given the session
prefix and measures ranking quality on the held-out next item(s).

In [3]:
train_data = diginetica.load_train()
val_data = diginetica.load_val()
test_data = diginetica.load_test()

next_item_eval = NextItemEvaluation.from_splits(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    exclude_unknowns=True,
    verbose=True,
    fmt="USIT",
)

Data from https://static.preferred.ai/cornac/datasets/diginetica/train.zip
will be cached into /root/.cornac/diginetica/train.csv


0.00B [00:00, ?B/s]

Unzipping ...
File cached!
Data from https://static.preferred.ai/cornac/datasets/diginetica/val_sbr.zip
will be cached into /root/.cornac/diginetica/val_sbr.csv


0.00B [00:00, ?B/s]

Unzipping ...
File cached!
Data from https://static.preferred.ai/cornac/datasets/diginetica/test_sbr.zip
will be cached into /root/.cornac/diginetica/test_sbr.csv


0.00B [00:00, ?B/s]

Unzipping ...
File cached!
rating_threshold = 1.0
exclude_unknowns = True
---
Training data:
Number of users = 571
Number of items = 4199
Number of sessions = 1528
---
Test data:
Number of users = 571
Number of items = 4199
Number of sessions = 416
Number of unknown users = 0
Number of unknown items = 0
---
Validation data:
Number of users = 571
Number of items = 4199
Number of sessions = 451
---
Total users = 571
Total items = 4199
Total sessions = 2395


## 3. Training the five models

A few words on each encoder:

- **GRU4Rec** (Hidasi et al., 2015) runs a GRU over the session and predicts from
  the last hidden state, the original session-based RNN.
- **FPMC** (Rendle et al., 2010) factorizes a personalized *first-order* Markov
  chain: it only looks at the **last** clicked item plus a user factor.
- **SASRec** (Kang & McAuley, 2018) is a left-to-right self-attention stack; the
  last position attends over the whole session prefix.
- **BERT4Rec** (Sun et al., 2019) uses a *bidirectional* Transformer (a HuggingFace
  BERT backbone).
- **GPT2Rec** (inspired by de Souza Pereira Moreira et al., 2021) uses a *causal* Transformer (a
  HuggingFace GPT-2 backbone).

All five share the **same scoring head** (dot-product of the session representation
with item embeddings) and the **same losses**, so differences below come from the
encoder and the chosen loss/learning-rate. The hyperparameters here are the best
configs found by the sweep in Section 5 (epochs are trimmed so the notebook runs
quickly; bump them up to reproduce the paper-level numbers). We add `SPop` (predict
the most popular items in the session) as a trivial baseline.

In [4]:
# Shared transformer config (best-on-validation by NDCG@10).
transformer = dict(
    embedding_dim=64,
    loss="cross-entropy",
    n_sample=512,
    batch_size=128,
    n_epochs=100,
    max_len=20,
    num_blocks=2,
    num_heads=2,
    model_selection="best",
    val_eval_every=5,
    val_metric="ndcg",
    val_k=10,
    device=DEVICE,
    verbose=True,
    seed=SEED,
)

models = [
    SPop(),
    GRU4Rec(
        layers=[100],
        loss="cross-entropy",
        dropout_p_hidden=0.3,
        sample_alpha=0.75,
        n_sample=512,
        batch_size=64,
        learning_rate=0.1,
        n_epochs=100,
        model_selection="best",
        val_eval_every=5,
        val_metric="recall",
        val_k=20,
        device=DEVICE,
        verbose=True,
        seed=SEED,
    ),
    FPMC(
        embedding_dim=64,
        loss="cross-entropy",
        n_sample=512,
        batch_size=128,
        learning_rate=0.1,
        n_epochs=100,
        model_selection="best",
        val_eval_every=5,
        val_metric="ndcg",
        val_k=10,
        device=DEVICE,
        verbose=True,
        seed=SEED,
    ),
    SASRec(learning_rate=0.01, **transformer),
    BERT4Rec(learning_rate=0.01, **transformer),
    GPT2Rec(learning_rate=0.001, **transformer),
]

metrics = [NDCG(k=10), NDCG(k=50), Recall(k=10), Recall(k=50), MRR()]

In [5]:
cornac.Experiment(
    eval_method=next_item_eval,
    models=models,
    metrics=metrics,
).run()


[SPop] Training started!

[SPop] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


[GRU4Rec] Training started!


  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]


[GPT2Rec] Evaluation started!


Ranking:   0%|          | 0/416 [00:00<?, ?it/s]

Ranking:   0%|          | 0/451 [00:00<?, ?it/s]


VALIDATION:
...
         |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Time (s)
-------- + ------ + ------- + ------- + --------- + --------- + --------
SPop     | 0.2376 |  0.2490 |  0.2590 |    0.2926 |    0.3376 |   2.6797
GRU4Rec  | 0.3325 |  0.3710 |  0.3795 |    0.4984 |    0.5370 |   0.6871
FPMC     | 0.1834 |  0.2117 |  0.2300 |    0.3183 |    0.3987 |   0.5353
SASRec   | 0.3331 |  0.3718 |  0.3821 |    0.5016 |    0.5466 |   1.7112
BERT4Rec | 0.3377 |  0.3731 |  0.3832 |    0.4920 |    0.5370 |   1.2461
GPT2Rec  | 0.3425 |  0.3698 |  0.3859 |    0.4662 |    0.5402 |   1.1649

TEST:
...
         |    MRR | NDCG@10 | NDCG@50 | Recall@10 | Recall@50 | Train (s) | Test (s)
-------- + ------ + ------- + ------- + --------- + --------- + --------- + --------
SPop     | 0.2344 |  0.2432 |  0.2506 |    0.2776 |    0.3110 |    0.0028 |   1.3359
GRU4Rec  | 0.2917 |  0.3284 |  0.3399 |    0.4515 |    0.5017 |   72.6985 |   0.6341
FPMC     | 0.1752 |  0.2005 |  0.2173 |    0.297

## 4. Loss functions

Every model is trained with a `(B, B+N)` score matrix: row *i* holds the score of
session *i* against the **B in-batch positives** (the diagonal is its true next
item) plus **N shared sampled negatives**. The `loss` argument picks how that
matrix is turned into a scalar:

- **`cross-entropy`** — softmax over all columns, maximize the diagonal
  (a.k.a. sampled softmax / `xe_softmax`; `ce` is the same objective).
- **`bpr`** — Bayesian Personalized Ranking; pairwise log-sigmoid of
  *(positive − negative)*.
- **`bpr-max`** — BPR-max (Hidasi & Karatzoglou, 2018): softmax-weighted negatives
  with a score regularizer.
- **`top1`** — the TOP1 ranking loss from the original GRU4Rec paper.
- **`bce`** — binary cross-entropy treating the diagonal as 1 and every other
  column as 0.

Swapping the loss is a one-argument change, e.g. `SASRec(loss="bpr", ...)`.

## 5. Which model × loss works best?

Running the full grid live would take too long, so we load a **pre-computed sweep**
(`tuning_results.csv`): every (model, loss) pair tuned over its learning rate on the
validation set, reporting the held-out **test** metrics of the best config. The CSV
is produced by `tasks/exp_diginetica/tune_all.py`.

In [6]:
cache("http://static.preferred.ai/cornac/datasets/diginetica/tuning_results.csv", unzip=False)

Data from http://static.preferred.ai/cornac/datasets/diginetica/tuning_results.csv
will be cached into /root/.cornac/tuning_results.csv


0.00B [00:00, ?B/s]

File cached!


'/root/.cornac/tuning_results.csv'

In [12]:
CSV_PATH = "/root/.cornac/tuning_results.csv"

results = pd.read_csv(CSV_PATH)
best = results.sort_values("val_NDCG@10", ascending=False).groupby(["model", "loss"], as_index=False).first()
best[
    ["model", "loss", "learning_rate", "test_NDCG@10", "test_NDCG@50", "test_Recall@10", "test_Recall@50", "test_MRR"]
]
# leaderboard = best.sort_values("test_NDCG@10", ascending=False).reset_index(drop=True)

,model,loss,learning_rate,test_NDCG@10,test_NDCG@50,test_Recall@10,test_Recall@50,test_MRR
0,BERT4Rec,bce,0.0030,0.3483,0.3675,0.4749,0.5585,0.3126
1,BERT4Rec,bpr,0.0100,0.3704,0.3814,0.4749,0.5251,0.3402
2,BERT4Rec,bpr-max,0.0010,0.3682,0.3798,0.4482,0.5017,0.3454
3,BERT4Rec,cross-entropy,0.0030,0.3607,0.3747,0.4716,0.5318,0.3292
4,BERT4Rec,top1,0.0010,0.3555,0.3632,0.4482,0.4816,0.3281
5,FPMC,bce,0.2000,0.0177,0.0241,0.0301,0.0602,0.0170
6,FPMC,bpr,0.1500,0.1630,0.1792,0.2542,0.3278,0.1395
7,FPMC,bpr-max,0.1500,0.2000,0.2073,0.2776,0.3110,0.1782
8,FPMC,cross-entropy,0.2000,0.2134,0.2257,0.2910,0.3445,0.1927
9,FPMC,top1,0.0200,0.2216,0.2437,0.3177,0.4114,0.1978


In [17]:
# Model x loss heatmap of test NDCG@10 (higher = better).
pivot = best.pivot(index="model", columns="loss", values="test_NDCG@10")
model_order = pivot.max(axis=1).sort_values(ascending=False).index
loss_order = pivot.max(axis=0).sort_values(ascending=False).index
pivot = pivot.loc[model_order, loss_order]
pivot.style.background_gradient(cmap="Greens", axis=None).format("{:.4f}")

loss,bce,cross-entropy,bpr,top1,bpr-max
model,,,,,
SASRec,0.3746,0.3739,0.3723,0.3619,0.3688
BERT4Rec,0.3483,0.3607,0.3704,0.3555,0.3682
GPT2Rec,0.3569,0.3673,0.3691,0.3694,0.3668
GRU4Rec,0.0146,0.3510,0.3156,0.3482,0.1501
FPMC,0.0177,0.2134,0.1630,0.2216,0.2000


Read this table two ways: **down a column** to see which encoder suits a given loss,
and **across a row** to see how sensitive a model is to its loss. The single best
(model, loss) cell is the top row of the leaderboard above.

## References

1. Hidasi, B., Karatzoglou, A., Baltrunas, L., & Tikk, D. (2015). *Session-based
   Recommendations with Recurrent Neural Networks.* arXiv:1511.06939.
2. Rendle, S., Freudenthaler, C., & Schmidt-Thieme, L. (2010). *Factorizing
   Personalized Markov Chains for Next-Basket Recommendation.* WWW.
3. Kang, W.-C., & McAuley, J. (2018). *Self-Attentive Sequential Recommendation.*
   ICDM. arXiv:1808.09781.
4. Sun, F., et al. (2019). *BERT4Rec: Sequential Recommendation with Bidirectional
   Encoder Representations from Transformer.* CIKM. arXiv:1904.06690.
5. de Souza Pereira Moreira, G., et al. (2021). *Transformers4Rec: Bridging the Gap
   between NLP and Sequential / Session-Based Recommendation.* RecSys.
6. Hidasi, B., & Karatzoglou, A. (2018). *Recurrent Neural Networks with Top-k Gains
   for Session-based Recommendations.* CIKM.